# 1.1) Variables, Data Types, Operators and File I/O

This notebook builds the smallest complete toolkit for handling environmental measurements in python: variables, the core scalar types, arithmetic, casting, string formatting, and reading and writing text files. We work throughout with one running example, a week of daily mean air-temperature readings from an alpine station, and finish by showing how a single type confusion can make generated code return the wrong answer without raising an error.

:::{admonition} Learning objectives
:class: tip
- Create variables whose names encode the physical quantity and its unit.
- Distinguish python's core scalar types and inspect them with type().
- Apply arithmetic, comparison, and casting to environmental measurements.
- Format numbers and units cleanly with f-strings.
- Read and write small text files with pathlib, and cast text back to numbers.
- Recognise why values read from files are strings, and the silent bugs that follow.
:::

## Variables and the core scalar types

A variable binds a name to a value, and python infers the value's *type*, which determines what operations are allowed. Five scalar types cover almost everything a single measurement needs: `int` (whole counts), `float` (real-valued measurements), `str` (text), `bool` (true/false flags), and `None` (a deliberate "no value", useful for missing or not-yet-assigned information).

In [1]:
# a single air-temperature reading at an alpine station
temp_celsius = -2.3            # float: degrees Celsius
station_name = "Jungfraujoch"  # str
n_readings = 7                 # int: number of daily means
is_subzero = temp_celsius < 0  # bool
quality_flag = None            # None: quality information not yet assigned

print(temp_celsius, station_name, n_readings, is_subzero, quality_flag)
print(type(temp_celsius), type(n_readings), type(station_name), type(is_subzero))

-2.3 Jungfraujoch 7 True None
<class 'float'> <class 'int'> <class 'str'> <class 'bool'>


:::{admonition} Computational-thinking fundamental: a name is a contract
:class: important
A variable name should state the physical quantity and, where it has one, the unit. `temp_celsius` and `temp_kelvin` cannot be confused; `t1` and `t2` can. Encoding the unit in the name turns a whole class of unit errors into something you can see while reading the code, before it ever runs.
:::

## Arithmetic, comparison, and casting

Arithmetic (`+ - * / // % **`) and comparison (`< <= > >= == !=`) operators behave as in mathematics, with two things to watch: `/` always returns a `float`, and operators are *type-dependent* — `+` adds numbers but concatenates strings. Casting converts between types with `int()`, `float()`, and `str()`.

In [2]:
# arithmetic: unit conversions, with units encoded in the names
temp_kelvin = temp_celsius + 273.15
temp_fahrenheit = temp_celsius * 9 / 5 + 32
print(temp_kelvin, temp_fahrenheit)

# division: / is always float; // floors; % is the remainder
total_days = 7
warm_days = 3
fraction_warm = warm_days / total_days
print(fraction_warm, warm_days // total_days, warm_days % total_days)

# comparison operators return bool
print(temp_kelvin > 270.0, temp_celsius != 0.0)

270.84999999999997 27.86
0.42857142857142855 0 3
True True


In [3]:
# values read from text arrive as str and must be cast before arithmetic
raw = "4.7"                       # a discharge reading, m^3 s^-1, as text
discharge_m3s = float(raw)
print(discharge_m3s + 1.0)        # works: now a float

# casting between types
count_str = str(n_readings)
print(count_str + " readings")    # str + str concatenates
print(int(3.99))                  # truncates toward zero -> 3, not 4

5.7
7 readings
3


:::{admonition} Common mistake: int() truncates, it does not round
:class: warning
`int(3.99)` is `3`, not `4`: casting a float to int discards the fractional part toward zero. Use `round()` for nearest-integer behaviour, but note that `round()` uses round-half-to-even (`round(2.5)` is `2`), which is correct for unbiased aggregation but surprising if you expect schoolbook rounding.
:::

## Formatting numbers with f-strings

f-strings (`f"..."`) interpolate variables and apply a format specifier after a colon: `:.2f` fixes two decimals, `:.3e` uses scientific notation, `:.1%` formats a fraction as a percentage. This keeps reported precision honest and units explicit.

In [4]:
# f-strings interpolate values and apply format specifiers after ':'
print(f"{station_name}: {temp_celsius:.1f} °C ({temp_kelvin:.2f} K)")
print(f"discharge = {discharge_m3s:.3e} m^3 s^-1")
print(f"warm fraction = {fraction_warm:.1%}")

Jungfraujoch: -2.3 °C (270.85 K)
discharge = 4.700e+00 m^3 s^-1
warm fraction = 42.9%


:::{admonition} Quick exercise: convert and report
:class: note
Given `temp_celsius = -2.3`, compute the temperature in kelvin and in degrees Fahrenheit, then print one line reporting all three to one decimal place with their units.
:::

:::{admonition} Solution
:class: note dropdown
```python
temp_celsius = -2.3
temp_kelvin = temp_celsius + 273.15
temp_fahrenheit = temp_celsius * 9 / 5 + 32
print(f"{temp_celsius:.1f} °C = {temp_kelvin:.1f} K = {temp_fahrenheit:.1f} °F")
```
:::

## Working with text: string methods

Measurements often arrive embedded in text — station records, log lines, CSV fields. String methods clean and split them: `.strip()` removes surrounding whitespace, `.split(sep)` breaks a string into parts, and `.lower()`/`.upper()`/`.title()` normalise case. The parts are still text and must be cast before arithmetic.

In [5]:
# a messy station record: extra spaces, mixed case, semicolon-separated
record = "  JUNGFRAUJOCH ; -2.3 ; degC  "
fields = record.split(";")
name = fields[0].strip().title()
value_celsius = float(fields[1].strip())
unit = fields[2].strip().lower()
print(name, value_celsius, unit)

Jungfraujoch -2.3 degc


## Files and paths with pathlib

`pathlib.Path` builds filesystem paths in an os-independent way (no hand-written `/` or `\`), and its `.write_text()` / `.read_text()` methods cover small text files in one call. The loop below is the first place iteration appears; control flow is treated properly in the next subchapter.

In [6]:
from pathlib import Path

data_path = Path("air_temperature_week.txt")

# a week of daily mean air temperature, °C
readings_celsius = [-2.3, -1.1, 0.4, 1.2, -0.8, -3.1, -1.9]

# write one value per line (str() converts each float to text)
lines = []
for value in readings_celsius:
    lines.append(str(value))
data_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

# read back: the lines are strings, not floats
raw_lines = data_path.read_text(encoding="utf-8").splitlines()
print(raw_lines)

['-2.3', '-1.1', '0.4', '1.2', '-0.8', '-3.1', '-1.9']


:::{admonition} The key fact about file input
:class: note
Everything read from a text file is `str`. The lines above print with quotes — `'-2.3'`, not `-2.3` — because they are text. Arithmetic or comparison on them without casting is either an error or, worse, silently wrong. The next section shows the silently-wrong case.
:::

## When generated code lies: a type bug you can run

ai assistants produce code that runs and looks plausible. The dangerous failures are the ones that do not raise an error but return a wrong answer on some data. Here is a function, as a chatbot might return it, to find the warmest reading in a file.

In [7]:
def warmest_reading(path):
    # find the warmest reading in a file (as an ai assistant returned it)
    values = Path(path).read_text(encoding="utf-8").splitlines()
    hottest = values[0]
    for v in values:
        if v > hottest:
            hottest = v
    return hottest

print(warmest_reading(data_path))

1.2


On the alpine week it returns `1.2`, which is correct — the warmest day was 1.2 °C. The bug is invisible here. Now run the same function on a warmer week with two-digit values.

In [8]:
summer_path = Path("air_temperature_summer.txt")
summer_celsius = [8.4, 9.5, 10.2, 7.1, 9.9]

lines = []
for value in summer_celsius:
    lines.append(str(value))
summer_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(warmest_reading(summer_path))

9.9


:::{admonition} Diagnosis: lexicographic comparison
:class: warning
The function returns `9.9`, but the warmest reading is `10.2`. The values are strings, and `>` on strings compares them character by character (lexicographically): `'9.9' > '10.2'` is `True` because `'9'` comes after `'1'`. The negative sign hid the bug on the alpine data, because `'-'` sorts before any digit. The fix is to compare numbers, not text: cast each line with `float()` before comparing.
:::

In [9]:
def warmest_reading_celsius(path):
    # compare numbers, not text
    values = Path(path).read_text(encoding="utf-8").splitlines()
    hottest_celsius = float(values[0])
    for v in values:
        reading_celsius = float(v)
        if reading_celsius > hottest_celsius:
            hottest_celsius = reading_celsius
    return hottest_celsius

print(warmest_reading_celsius(summer_path), "°C")

10.2 °C


:::{admonition} Quick exercise: mean temperature, done right
:class: note
Read `air_temperature_week.txt`, cast each line to `float`, and compute the mean in °C, printed to two decimals. Use only built-in features.
:::

:::{admonition} Solution
:class: note dropdown
```python
raw_lines = data_path.read_text(encoding="utf-8").splitlines()
total_celsius = 0.0
for line in raw_lines:
    total_celsius += float(line)
mean_celsius = total_celsius / len(raw_lines)
print(f"{mean_celsius:.2f} °C")
```
:::

:::{admonition} Going deeper: reproducible environments with uv
:class: seealso dropdown
`uv` is a fast environment and dependency manager. A minimal reproducible project:

```bash
uv init my-analysis        # create a project with pyproject.toml
cd my-analysis
uv add numpy xarray        # add dependencies, updates uv.lock
uv run python analysis.py  # run inside the locked environment
```

`uv.lock` pins exact versions, so the environment reproduces on another machine. Creating and activating a bare virtual environment differs by shell:

```bash
# macOS (zsh) and Linux (bash)
uv venv
source .venv/bin/activate
```

```powershell
# Windows (PowerShell)
uv venv
.venv\Scripts\Activate.ps1
```
:::

:::{admonition} Going deeper: floating-point precision
:class: seealso dropdown
Floats are binary approximations of real numbers, so familiar decimals are not stored exactly:

```python
print(0.1 + 0.2)                       # 0.30000000000000004
print(0.1 + 0.2 == 0.3)                # False
import math
print(math.isclose(0.1 + 0.2, 0.3))    # True
```

Never test floats for equality with `==`; use `math.isclose` (or `numpy.isclose` for arrays) with an explicit tolerance. This matters whenever you compare or threshold measured quantities.
:::

:::{admonition} Going deeper: type hints and docstrings
:class: seealso dropdown
Annotations declare the intended types and a one-line description states the contract. Neither is enforced at runtime, but both make unit assumptions explicit and are checkable with tools such as mypy or ty.

```python
def celsius_to_kelvin(temp_celsius: float) -> float:
    # convert degrees Celsius to kelvin (absolute temperature)
    return temp_celsius + 273.15
```
:::

:::{admonition} Takeaways
:class: danger
- Every value has a type, and the type drives behaviour — `+` adds numbers but concatenates strings.
- Data read from a text file is text; cast with `float()`/`int()` before arithmetic or comparison.
- String comparison is lexicographic and silently wrong for numbers (`'9.9' > '10.2'` is `True`).
- A variable name is a contract: encode the quantity and unit (`temp_celsius`, `discharge_m3s`).
- Use `pathlib` for os-independent paths, and never compare floats with `==`.
:::

## Resources

- [Project Pythia Foundations](https://foundations.projectpythia.org/) — geoscience-flavoured tutorials on the core scientific-python stack; the natural next step after this chapter.
- [The Python tutorial](https://docs.python.org/3/tutorial/) — the authoritative reference for the built-in types and syntax used above.
- [uv documentation](https://docs.astral.sh/uv/) — the environment and dependency manager shown in the Going-deeper box on reproducible environments.